# Limpieza y normalización de entidades con variantes

Esta etapa crea una copia normalizada del JSON y del CSV de entidades agrupadas por variantes. No modifica los archivos originales: lee desde `data/80 json csv excel - variantes - todas las entidades/` y escribe archivos nuevos en `data/limpieza/`.

## Por qué se agrega `valor_referencia`

`valor_referencia` permite tener un valor normalizado y comparable para cada entidad agrupada. Sirve para revisar, buscar, deduplicar o preparar futuras etapas de extracción sin perder las variantes originales detectadas en el texto.

La diferencia clave es:

- `valor`: valor original de cada variante, conservado sin cambios.
- `valor_referencia`: valor normalizado elegido para representar a la entidad agrupada.

## Reglas aplicadas

- `persona`: minúsculas y espacios sobrantes removidos, sin eliminar títulos, roles ni palabras del nombre.
- `DNI`, `CUIT_CUIL`, `CBU`, `CVU`: solo dígitos.
- `MONTO`: solo dígitos, sin convertir a número decimal.
- `ALIAS`: minúsculas, espacios sobrantes removidos y puntos conservados como parte del alias.

Para cada entidad agrupada se normalizan todas sus variantes y se elige como `valor_referencia` el valor normalizado más largo.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_DIR = PROJECT_ROOT / "data" / "80 json csv excel - variantes - todas las entidades"
OUTPUT_DIR = PROJECT_ROOT / "data" / "limpieza"

print("Carpeta de entrada:", INPUT_DIR)
print("Carpeta de salida:", OUTPUT_DIR)

In [ ]:
json_files = sorted(INPUT_DIR.glob("*.json"))
csv_files = sorted(INPUT_DIR.glob("*.csv"))

print("JSON encontrados:")
for path in json_files:
    print("-", path.name)

print("CSV encontrados:")
for path in csv_files:
    print("-", path.name)

## Ejecutar la etapa reproducible

La lógica principal está en `src/limpieza_normalizacion_entidades_variantes.py` para evitar duplicar reglas entre notebook y script. Por defecto no sobrescribe salidas existentes.

In [ ]:
import subprocess
import sys

script = PROJECT_ROOT / "src" / "limpieza_normalizacion_entidades_variantes.py"
result = subprocess.run(
    [sys.executable, str(script)],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

## Salidas esperadas

- `data/limpieza/embargos_entidades_variantes_normalizado.json`
- `data/limpieza/embargos_entidades_variantes_normalizado.csv`

Opcionalmente, el script puede generar XLSX con `--with-xlsx` si `openpyxl` está instalado.